# AI003 — Trabalho Final: Amostragem, Intervalos de Confiança e Testes de Hipóteses
### Eleições 2026 em Porto Alegre: como mudou a preferência para Presidente entre o 1º e o 2º turno

Notebook de **organização e desenvolvimento** do trabalho final (base: `trabalho_final.txt`).

**Objetivo:** aplicar amostragem, estimação por intervalos de confiança e testes de hipóteses em uma grande base de dados, comparando resultados da **população** com os de uma **amostra** (AAS), com 3 painéis interativos + vídeo de até 5 min.

## Índice
1. [Checklist e cronograma](#1.-Checklist-e-cronograma)
2. [Base de dados](#2.-Base-de-dados)
3. [Dimensionamento e seleção da amostra](#3.-Dimensionamento-e-seleção-da-amostra)
4. [Painel 1 — População](#4.-Painel-1-—-Análise-descritiva-da-população)
5. [Painel 2 — Amostra](#5.-Painel-2-—-Análise-descritiva-da-amostra)
6. [Painel 3 — Inferência e comparação](#6.-Painel-3-—-Inferência-e-comparação)
7. [Roteiro do vídeo](#7.-Roteiro-do-vídeo-(máx.-5-min))
8. [Entrega e avaliação](#8.-Entrega-e-autoavaliação)

## 1. Checklist e cronograma

| # | Etapa | Peso | Status | Prazo |
|---|-------|------|--------|-------|
| 2 | Escolha, descrição e contextualização da base | 10% | ⬜ | |
| 3 | Dimensionamento e seleção da AAS | 10% | ⬜ | |
| 4 | Painel 1 — população | 10% | ⬜ | |
| 5 | Painel 2 — amostra | 10% | ⬜ | |
| 6 | ICs e testes (médias e proporções) | 20% | ⬜ | |
| 6 | Interpretação e comparação pop. × amostra | 20% | ⬜ | |
| 7 | Vídeo (≤ 5 min) | 20% | ⬜ | |
| 8 | Entrega (links públicos) | — | ⬜ | |

Legenda: ⬜ a fazer · 🟨 em andamento · ✅ concluído

### Decisões tomadas (preencher ao longo do trabalho)
- Ferramenta de dashboard: **Streamlit** (`dashboard/app.py`), com uma aba por painel (/dados, /populacao, /amostra, /inferencia) — precisa ser publicado com acesso público
- Base escolhida: Boletins de Urna (BU) das Eleições 2026, 1º e 2º turnos (TSE), recorte **Porto Alegre / Presidente** (N = 3.077 seções)
- Semente aleatória: 42

## 2. Base de dados

**Tema:** *como se construiu a mudança de preferência para Presidente em Porto Alegre entre o 1º e o 2º turno de 2026?*

**Requisitos:** base grande, fonte confiável, com variáveis **quantitativas** (média) e **qualitativas** (proporção).

### Ficha da base
| Item | Resposta |
|------|----------|
| Fonte e link de acesso | Tribunal Superior Eleitoral (TSE) — Portal de Dados Abertos, *Resultados 2026 — Boletim de Urna* (https://dadosabertos.tse.jus.br). Arquivos: `bweb_1t_RS_051020261403.csv` (1º turno, gerado em 05/10/2026) e `bweb_2t_RS_261020260742.csv` (2º turno, gerado em 26/10/2026). **Confirmar o link exato da página do conjunto de dados antes da entrega.** |
| Contexto e objetivo da base | Resultado oficial da votação por **boletim de urna (BU)**, isto é, por seção eleitoral, nas eleições gerais de 2026 (1º turno em 04/10/2026; 2º turno em 25/10/2026). Cada BU registra aptos, comparecimento, abstenções e votos de cada votável (candidato, legenda, branco, nulo). |
| Nº de observações (arquivo bruto, RS) | 1º turno: **3.649.489 linhas**; 2º turno: **106.789 linhas** (formato longo: 1 linha = seção × cargo × votável) |
| Nº de variáveis | **46 colunas** por arquivo (leiaute no `_leiame-boletimurnaweb.pdf`) |
| Cobertura do arquivo | RS inteiro: 497 municípios, 522 zonas, 27.547 seções (todas "Apurada"), ~8,52 milhões de eleitores aptos |
| **Recorte do trabalho** | **Município de Porto Alegre**, cargo **Presidente** (1º e 2º turnos) |
| **População de interesse** | As **3.077 seções eleitorais de Porto Alegre** (10 zonas, ~1,06 milhão de eleitores aptos), presentes nos dois turnos |
| **Unidade amostral** | **Seção eleitoral (urna)**, com 1 observação por seção após agregar o formato longo; a chave é `NR_ZONA + NR_SECAO` |
| Formato do arquivo | CSV em `latin1`, separador `;`, campos entre aspas, `#NULO#` e `-1` para ausentes (conforme o LEIAME) |
| **Base tratada (usada na análise)** | **3.077 linhas × 36 colunas**: 1 linha por seção, com as colunas do 1º turno (`_1t`), do 2º turno (`_2t`) e as derivadas. Salva em `dashboard/dados/populacao.csv`; a amostra (793 × 36) em `dashboard/dados/amostra.csv` |

> ⚠️ O arquivo é longo (1 linha por votável). Para a AAS e as estatísticas, é preciso **agregá-lo em 1 linha por seção**. Fazer a AAS sobre as linhas brutas violaria a igualdade de probabilidade entre unidades.

### Variáveis selecionadas
Ao agregar, cada seção tem os valores dos dois turnos (sufixos `_1t`, `_2t`), o que torna as comparações **pareadas**.

| Variável | Tipo | Descrição | Uso |
|----------|------|-----------|-----|
| `PCT_LULA_1t`, `PCT_LULA_2t` | quantitativa contínua (derivada) | % de votos válidos (nominais) de Lula na seção (1º turno: entre todos os candidatos; 2º turno: contra Flávio Bolsonaro) | média, descritiva |
| `PCT_LULA_TOTAL_1t`, `PCT_LULA_TOTAL_2t` | quantitativa contínua (derivada) | % de Lula sobre os **votos totais** (válidos + brancos + nulos) | média, descritiva |
| `PCT_BRANCO_NULO_1t`, `PCT_BRANCO_NULO_2t` | quantitativa contínua (derivada) | % de brancos e nulos sobre os votos totais | média, descritiva |
| `DELTA_LULA_PP` | quantitativa contínua (derivada) | `PCT_LULA_2t − PCT_LULA_1t`, em pontos percentuais | **média** (teste t pareado) |
| `PCT_ABSTENCAO_1t`, `PCT_ABSTENCAO_2t` | quantitativa contínua (derivada) | Abstenções / aptos da seção, em % | **média** |
| `DELTA_ABST_PP` | quantitativa contínua (derivada) | Variação da abstenção entre os turnos | média (apoio) |
| `QT_APTOS_1t` | quantitativa discreta | Eleitores aptos na seção | descritiva / controle |
| `VENCEDOR_1t`, `VENCEDOR_2t` | qualitativa nominal | Candidato mais votado na seção em cada turno (EMPATE quando Lula = Flávio) | **proporção** |
| `MUDOU_VENCEDOR` | qualitativa nominal (derivada) | Sim / Não / Indefinido (empate em algum turno) | **proporção** |
| `TRAJETORIA` | qualitativa nominal (derivada) | Ex.: "FLAVIO → LULA" | frequências, McNemar |
| `NR_ZONA` | qualitativa nominal | Zona eleitoral (10 zonas) | segmentação nos painéis |
| `NM_VOTAVEL`, `QT_VOTOS` | quali/quanti | Votável e votos recebidos | base para derivar vencedor e percentuais |

### Valores de referência da população (apurados no perfilamento; revalidar na carga)
| Indicador | 1º turno | 2º turno |
|-----------|----------|----------|
| Seções | 3.077 | 3.077 |
| Aptos | 1.064.200 | 1.063.955 |
| Lula (votos nominais) | 377.252 | 456.019 |
| Flávio Bolsonaro (votos nominais) | 325.139 | 360.690 |
| Seções vencidas por Lula | 68,1% (2.094) | 74,9% (2.304) |
| Seções vencidas por Flávio | 31,4% (966) | 24,4% (752) |
| Seções empatadas | 0,6% (17) | 0,7% (21) |
| Abstenção média por seção | 24,1% | 21,2% |

**Trajetória das seções:** 2.083 Lula → Lula · 740 Flávio → Flávio · **207 Flávio → Lula** · 9 Lula → Flávio · 38 com empate em algum turno (7,0% mudaram de vencedor entre os dois candidatos).
Variação média de `PCT_LULA` entre os turnos: **+7,1 p.p.** (desvio-padrão 2,2 p.p.).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

SEED = 42          # registrar para reprodutibilidade
ALPHA = 0.05       # nível de significância
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# Carregar e agregar para 1 linha por seção (unidade amostral) — apenas PORTO ALEGRE
import os
DIR1 = "bweb_1t_RS_051020261403/bweb_1t_RS_051020261403.csv"
DIR2 = "bweb_2t_RS_261020260742/bweb_2t_RS_261020260742.csv"
MUNICIPIO = "PORTO ALEGRE"
CARGO = "Presidente"
CHAVE = ["NR_ZONA", "NR_SECAO"]
COLS = ["NM_MUNICIPIO", "NR_ZONA", "NR_SECAO", "DS_CARGO_PERGUNTA", "QT_APTOS", "QT_COMPARECIMENTO",
        "QT_ABSTENCOES", "DS_TIPO_VOTAVEL", "NM_VOTAVEL", "QT_VOTOS"]

def agrega_turno(caminho, sufixo):
    pedacos = []
    for ch in pd.read_csv(caminho, sep=";", encoding="latin1", usecols=COLS, chunksize=500_000):
        pedacos.append(ch[(ch.NM_MUNICIPIO == MUNICIPIO) & (ch.DS_CARGO_PERGUNTA == CARGO)])
    d = pd.concat(pedacos)
    base = d[CHAVE + ["QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES"]].drop_duplicates(CHAVE).set_index(CHAVE)
    nom = d[d.DS_TIPO_VOTAVEL == "Nominal"]
    votos = nom.pivot_table(index=CHAVE, columns="NM_VOTAVEL", values="QT_VOTOS", aggfunc="sum", fill_value=0)
    out = base.join(votos[["LULA", "FLAVIO BOLSONARO"]])
    out["VOTOS_VALIDOS"] = votos.sum(axis=1)                       # votos nominais (válidos) do cargo
    out["QT_BRANCO_NULO"] = d[d.DS_TIPO_VOTAVEL.isin(["Branco", "Nulo"])].groupby(CHAVE).QT_VOTOS.sum()
    out["QT_BRANCO_NULO"] = out.QT_BRANCO_NULO.fillna(0)
    total = out.VOTOS_VALIDOS + out.QT_BRANCO_NULO                 # votos totais = válidos + brancos + nulos
    out["VENCEDOR"] = np.select([out.LULA > out["FLAVIO BOLSONARO"], out["FLAVIO BOLSONARO"] > out.LULA],
                               ["LULA", "FLAVIO BOLSONARO"], default="EMPATE")   # empate é categoria própria
    out["PCT_LULA"] = out.LULA / out.VOTOS_VALIDOS * 100              # % dos votos válidos
    out["PCT_FLAVIO"] = out["FLAVIO BOLSONARO"] / out.VOTOS_VALIDOS * 100
    out["PCT_LULA_TOTAL"] = out.LULA / total * 100                    # % sobre votos totais (com brancos e nulos)
    out["PCT_FLAVIO_TOTAL"] = out["FLAVIO BOLSONARO"] / total * 100
    out["PCT_BRANCO_NULO"] = out.QT_BRANCO_NULO / total * 100
    out["PCT_ABSTENCAO"] = out.QT_ABSTENCOES / out.QT_APTOS * 100
    return out.add_suffix("_" + sufixo)

t1 = agrega_turno(DIR1, "1t")
t2 = agrega_turno(DIR2, "2t")
pop = t1.join(t2, how="inner").reset_index()   # 1 linha por seção de Porto Alegre

print("Observações (N = seções):", len(pop))
print("Variáveis:", pop.shape[1])
pop.info()
pop.head()

In [ ]:
# Variáveis derivadas e seleção para análise
pop["DELTA_LULA_PP"] = pop.PCT_LULA_2t - pop.PCT_LULA_1t             # variação em pontos percentuais (pareada)
pop["DELTA_LULA_TOTAL_PP"] = pop.PCT_LULA_TOTAL_2t - pop.PCT_LULA_TOTAL_1t
pop["DELTA_ABST_PP"] = pop.PCT_ABSTENCAO_2t - pop.PCT_ABSTENCAO_1t
empate = (pop.VENCEDOR_1t == "EMPATE") | (pop.VENCEDOR_2t == "EMPATE")
pop["MUDOU_VENCEDOR"] = np.select([empate, pop.VENCEDOR_1t != pop.VENCEDOR_2t],
                                  ["Indefinido (empate)", "Sim"], default="Não")
pop["PCT_OUTROS_1t"] = 100 - pop.PCT_LULA_1t - pop.PCT_FLAVIO_1t
pop["TRAJETORIA"] = pop.VENCEDOR_1t.str.split().str[0] + " → " + pop.VENCEDOR_2t.str.split().str[0]

VAR_QUANTI = ["QT_APTOS_1t", "PCT_LULA_1t", "PCT_LULA_2t", "DELTA_LULA_PP", "PCT_ABSTENCAO_1t", "PCT_ABSTENCAO_2t"]
VAR_QUALI = ["VENCEDOR_1t", "VENCEDOR_2t", "MUDOU_VENCEDOR", "TRAJETORIA", "NR_ZONA"]

pop[VAR_QUANTI].describe()

## 3. Dimensionamento e seleção da amostra

Ferramenta: **Calculadora de tamanho amostral — USP/Bauru** (sugerida no enunciado). O mesmo cálculo é reproduzido em Python na célula abaixo (`tamanho_amostra`), para conferência e reprodutibilidade.

O tamanho foi dimensionado para **estimar uma proporção** (o caso mais exigente com p = 0,5) e serve também para as médias: com desvio-padrão ≈ 2,2 p.p. em `DELTA_LULA_PP`, a margem de erro da média com n = 793 é ≈ 0,13 p.p.

### Parâmetros (justificar cada um)
| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| Nível de confiança | 95% | convenção em pesquisas eleitorais e em inferência (z = 1,96); coerente com α = 5% nos testes. É o equilíbrio entre confiança e custo: com 90% o n seria 605; com 99%, 1.153 (37% da população) |
| Margem de erro | 3 p.p. | mesma ordem das pesquisas eleitorais (2 a 3 p.p.) e menor que os efeitos de interesse (Lula passou de 68% para 75% das seções; ganho médio de ~7 p.p.), então a amostra consegue detectá-los. Alternativas: 2 p.p. → n = 1.349 (44% da população); 4 p.p. → 503; 5 p.p. → 342 |
| Proporção considerada (p) | 0,5 | p(1 − p) é máximo em p = 0,5, então o n vale para qualquer proporção estudada (seções vencidas ≈ 0,7; mudança de vencedor ≈ 0,07). Usar o valor da população seria tirar o parâmetro do próprio resultado |
| Tamanho da população (N) | 3.077 | seções de Porto Alegre |
| **n calculado** | **793** | n₀ = 1.067 sem correção; com a correção para população finita, n = 793 (fração amostral n/N = 25,8%). Como a fração passa de 5%, os ICs e testes do Painel 3 também devem usar o fator de correção √((N − n)/(N − 1)) = 0,862 |

Fórmula para proporção com população finita:
$$n_0=\frac{z^2\,p(1-p)}{e^2},\qquad n=\frac{n_0}{1+\frac{n_0-1}{N}}$$

In [ ]:
def tamanho_amostra(N, conf=0.95, erro=0.05, p=0.5):
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    n0 = z**2 * p * (1 - p) / erro**2
    return int(np.ceil(n0 / (1 + (n0 - 1) / N)))

n = tamanho_amostra(N=len(pop), conf=0.95, erro=0.03, p=0.5)   # N=3.077 → n=793
print("Tamanho da amostra:", n)

In [ ]:
# Amostragem Aleatória Simples (sem reposição): cada elemento tem a mesma probabilidade n/N
amostra = pop.sample(n=n, replace=False, random_state=SEED)
os.makedirs("dados", exist_ok=True)
amostra.to_csv("dados/amostra.csv", index=False)
print(len(amostra), "linhas | fração amostral:", round(len(amostra)/len(pop), 4))

### Procedimento da amostra aleatória simples (AAS)

1. **Cadastro (moldura amostral):** o DataFrame `pop`, com as 3.077 seções, uma por linha. A chave zona + seção é única (sem duplicatas), então cada seção aparece exatamente uma vez. A amostragem é feita **depois** da agregação: sortear linhas do arquivo bruto (uma por candidato votado) daria mais chance às seções com mais candidatos votados.
2. **Sorteio:** `pop.sample(n=793, replace=False, random_state=42)`. O pandas usa o gerador do NumPy com a semente 42 e sorteia 793 posições distintas entre as 3.077, o que equivale a numerar as seções de 1 a N e sortear 793 números sem reposição.
3. **Probabilidade igual:** sem reposição e com sorteio uniforme, todo subconjunto de 793 seções tem a mesma chance de ser escolhido. Logo, cada seção tem a mesma probabilidade de inclusão: n/N = 793/3.077 = **25,8%**.
4. **Reprodutibilidade:** mesma semente + mesma ordem de linhas = mesma amostra. Conferido: o sorteio refeito devolve as mesmas 793 seções, na mesma ordem, gravadas em `dashboard/dados/amostra.csv` (o script `dashboard/preparar_dados.py` usa a mesma lógica).
5. **Semente fixada antes da análise:** a semente 42 foi definida antes de calcular qualquer resultado e **não foi trocada** depois, mesmo com uma estimativa atípica (o IC de `DELTA_LULA_PP` não contém a média populacional). Trocar a semente para "melhorar" a amostra invalidaria a inferência.
6. **Checagem de representatividade:** a distribuição das seções por zona na amostra não difere da população (teste qui-quadrado de aderência: χ² = 6,0; valor-p = 0,74).

## 4. Painel 1 — Análise descritiva da população

Conteúdo mínimo:
- Indicadores (KPIs): N, médias, proporções principais
- Distribuições das variáveis selecionadas (histograma/boxplot para quanti; barras para quali)
- Frequências e proporções (quali)
- Média, mediana, desvio-padrão, quartis, mín/máx, CV (quanti)

In [ ]:
def resumo_quanti(df, cols):
    r = df[cols].agg(["count", "mean", "median", "std", "min", "max"]).T
    r["q1"] = df[cols].quantile(.25); r["q3"] = df[cols].quantile(.75)
    r["cv"] = r["std"] / r["mean"]
    return r

def resumo_quali(df, col):
    f = df[col].value_counts(dropna=False)
    return pd.DataFrame({"freq": f, "prop": f / f.sum()})

# resumo_quanti(pop, VAR_QUANTI)
# for c in VAR_QUALI: display(resumo_quali(pop, c))

In [ ]:
# Esboço de gráficos (reaproveitar no dashboard)
# fig, ax = plt.subplots(1, 2, figsize=(11, 4))
# sns.histplot(pop[VAR_QUANTI[0]], kde=True, ax=ax[0]); sns.boxplot(x=pop[VAR_QUANTI[0]], ax=ax[1])
# plt.show()

## 5. Painel 2 — Análise descritiva da amostra

Reproduzir **exatamente** o Painel 1, usando apenas a amostra, e destacar diferenças em distribuições e medidas (ex.: tabela lado a lado e gráficos sobrepostos).

In [ ]:
# comp = pd.concat({"populacao": resumo_quanti(pop, VAR_QUANTI),
#                   "amostra":   resumo_quanti(amostra, VAR_QUANTI)}, axis=1)
# comp

## 6. Painel 3 — Inferência e comparação

> Parâmetros populacionais: calculados **diretamente da população**. Inferência: usa **apenas a amostra**.

Mínimo exigido: **2 hipóteses para proporção** e **2 para média**.

### Roteiro por hipótese
1. Variável e evento de interesse
2. $H_0$ e $H_1$ (e justificativa do valor de referência)
3. Condições de aplicação
4. IC (confiança = 1 − α)
5. Estatística de teste, valor-p, α, decisão
6. Comparação com o valor populacional e interpretação

**Condições:** proporção → $n p_0\ge10$ e $n(1-p_0)\ge10$; média → normalidade aproximada ou $n$ grande (TCL), σ desconhecido ⇒ teste t.

### Registro das hipóteses
| ID | Parâmetro | Variável/evento | H0 | H1 | Valor de referência | Justificativa |
|----|-----------|-----------------|----|----|---------------------|---------------|
| P1 | proporção | `VENCEDOR_2t` = LULA | p = 0,6805 | p > 0,6805 | 0,6805 | resultado oficial do 1º turno (68,05% das seções), conhecido antes do 2º turno: Lula ganhou território? Amostra: 74,02% (IC 71,39–76,65); população: 74,88% |
| P2 | proporção | `MUDOU_VENCEDOR` = Sim | p = 0,05 | p > 0,05 | 0,05 | convenção para "fenômeno raro" (1 em 20 seções); a referência mais arbitrária. Empates contam como "não virou". Amostra: 8,07% (IC 6,44–9,70); população: 7,02% |
| M1 | média (pareada) | `DELTA_LULA_PP` | μ = 4,74 | μ > 4,74 | 4,74 p.p. | metade dos 9,47% dos demais candidatos no 1º turno (divisão igual): Lula herdou mais da metade? Contra 0 seria trivial. Amostra: 7,37 (IC 7,23–7,50); população: 7,15 — **IC não contém o real** |
| M2 | média (pareada) | `DELTA_ABST_PP` | μ = 0 | μ ≠ 0 | 0 p.p. | ausência de mudança; bilateral (abstenção poderia subir ou cair). Amostra: −2,94 (IC −3,05 a −2,82); população: −2,89 |

> **Hipóteses finais**, implementadas no Painel 3 do dashboard (com α = 5%, IC de 95% e correção para população finita). O valor de H₀ vem sempre de fora do parâmetro testado (1º turno, regra neutra ou convenção), nunca do resultado do 2º turno. Os 4 testes rejeitam H₀ (valor-p < 0,001); 3 dos 4 ICs contêm o parâmetro real.

In [ ]:
def inferencia_proporcao(amostra, col, evento, p0, alternativa="two-sided", alpha=ALPHA, pop=None):
    x = (amostra[col] == evento).sum(); n = len(amostra); p_hat = x / n
    ic = proportion_confint(x, n, alpha=alpha, method="normal")
    z, p = proportions_ztest(x, n, value=p0, alternative=alternativa, prop_var=p0)
    out = {"n": n, "p_hat": p_hat, "IC_inf": ic[0], "IC_sup": ic[1],
           "p0": p0, "z": z, "valor_p": p, "rejeita_H0": p < alpha,
           "cond_np0>=10": n*p0 >= 10, "cond_n(1-p0)>=10": n*(1-p0) >= 10}
    if pop is not None:
        out["p_populacional"] = (pop[col] == evento).mean()
        out["IC_contem_p_pop"] = ic[0] <= out["p_populacional"] <= ic[1]
    return pd.Series(out)

def inferencia_media(amostra, col, mu0, alternativa="two-sided", alpha=ALPHA, pop=None):
    x = amostra[col].dropna(); n = len(x)
    t, p = stats.ttest_1samp(x, mu0, alternative=alternativa)
    se = x.std(ddof=1) / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha/2, n - 1)
    out = {"n": n, "x_barra": x.mean(), "s": x.std(ddof=1),
           "IC_inf": x.mean() - tcrit*se, "IC_sup": x.mean() + tcrit*se,
           "mu0": mu0, "t": t, "valor_p": p, "rejeita_H0": p < alpha}
    if pop is not None:
        out["mu_populacional"] = pop[col].mean()
        out["IC_contem_mu_pop"] = out["IC_inf"] <= out["mu_populacional"] <= out["IC_sup"]
    return pd.Series(out)

# inferencia_proporcao(amostra, "coluna", "evento", p0=0.5, pop=pop)
# inferencia_media(amostra, "coluna", mu0=0, pop=pop)

In [ ]:
# Verificação de normalidade (apoio à condição da média)
# stats.probplot(amostra[VAR_QUANTI[0]], plot=plt); plt.show()
# stats.shapiro(amostra[VAR_QUANTI[0]].sample(min(5000, len(amostra)), random_state=SEED))

### Comparação e interpretação (preencher)
- **Diferença entre parâmetros populacionais e estimativas amostrais:**
- **Estimativa pontual × intervalo de confiança** (o IC contém o parâmetro populacional?):
- **Conclusões dos testes:**
- **Limitações e efeito da variabilidade amostral:** _(ex.: repetir a AAS várias vezes e mostrar a % de ICs que cobrem o valor verdadeiro)_

In [ ]:
# Extra opcional: cobertura dos ICs em várias amostras (ilustra variabilidade amostral)
# def cobertura(col, evento, B=500):
#     p_pop = (pop[col] == evento).mean(); ok = 0
#     for b in range(B):
#         s = pop.sample(n=n, random_state=b)
#         lo, hi = proportion_confint((s[col]==evento).sum(), n, alpha=ALPHA)
#         ok += lo <= p_pop <= hi
#     return ok / B

### Implementação dos painéis
- Ferramenta: Streamlit — `streamlit run dashboard/app.py` (abas: 📖 Dados, 🏙️ Painel 1, 🎲 Painel 2, 📐 Painel 3)
- Link Painel 1: _
- Link Painel 2: _
- Link Painel 3: _
- Interatividade (filtros/seletores): _
- Acesso público sem pedir autorização: ⬜

## 7. Roteiro do vídeo (máx. 5 min)

Priorizar **interpretação estatística**, não descrição de gráficos nem exposição de cálculos.

| Bloco | Conteúdo | Tempo alvo |
|-------|----------|-----------|
| 1 | Base, origem, população e unidade amostral | 0:30 |
| 2 | Tamanho da amostra (parâmetros) e seleção da AAS | 0:45 |
| 3 | Painéis 1 e 2: principais achados e diferenças pop. × amostra | 1:15 |
| 4 | Hipóteses, ICs e testes (proporção e média) | 1:30 |
| 5 | Interpretação, comparação, limitações | 1:00 |
| | **Total** | **5:00** |

Checklist: ensaiar com cronômetro · áudio claro · vídeo no YouTube (público/não listado) · link testado em janela anônima.

## 8. Entrega e autoavaliação

### Itens a entregar
- [ ] Link dos painéis (1, 2 e 3)
- [ ] Link do vídeo no YouTube (≤ 5 min)
- [ ] Link da base original + identificação da fonte
- [ ] Todos os links abertos sem solicitar autorização
- [ ] Painéis e vídeo disponíveis durante todo o período de avaliação

### Autoavaliação pelos critérios
- [ ] Base descrita (fonte, link, N, variáveis, população, unidade, variáveis) — 10%
- [ ] Parâmetros do cálculo de n justificados; AAS documentada — 10%
- [ ] Painel da população adequado aos tipos de variável — 10%
- [ ] Painel da amostra espelha o da população — 10%
- [ ] ICs e testes corretos, com condições verificadas — 20%
- [ ] Interpretação e comparação clara pop. × amostra — 20%
- [ ] Vídeo claro, objetivo, ≤ 5 min — 20%